# Week 08 - Thursday Assignment

**Topics:** RNNs, LSTMs, BPTT, Sequence Prediction

*Note on AI Usage:* Prompts used during the development of this notebook and the rationales behind them are documented in `prompts.md` OR at the bottom of the relevant cells.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

# Constants
STOCK_CSV_PATH = "Day 47 stock_prices.csv"
WINDOW_SIZE = 14 # 2 weeks of trading context
BATCH_SIZE = 32
RANDOM_SEED = 42

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    
set_seed(RANDOM_SEED)

## Sub-step 1: Sequence Dataset Construction (Easy)
We build a sequence dataset for the next-day close price prediction.

**Splitting Strategy:** Time-series data must be split chronologically (e.g., first 80% of dates for training, last 20% for testing). Using a random split (e.g., `train_test_split` with `shuffle=True`) causes data leakage because future prices leak into the training set, allowing the model to "cheat" by seeing future context. The reported performance would be artificially inflated and the model would fail in production.

**Window Size Justification:** A window size of 14 days is chosen to provide approximately two trading weeks of context.

In [ ]:
from typing import Tuple

def load_and_clean_stock_data(filepath: str, ticker: str = 'RELIANCE') -> pd.DataFrame:
    '''Loads stock data and filters for a specific ticker.'''
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"Error: {filepath} not found.")
        return pd.DataFrame()
        
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    df = df[df['ticker'] == ticker].copy()
    
    # Normalize the 'close' price for stable neural network training
    close_min = df['close'].min()
    close_max = df['close'].max()
    df['close_scaled'] = (df['close'] - close_min) / (close_max - close_min)
    
    return df, close_min, close_max

def create_stock_sequences(df: pd.DataFrame, window_size: int) -> Tuple[np.ndarray, np.ndarray]:
    '''Constructs sequence features (X) and next-day targets (y).'''
    X, y = [], []
    prices = df['close_scaled'].values
    
    if len(prices) <= window_size:
        raise ValueError("Dataset length is smaller than window size.")
        
    for i in range(len(prices) - window_size):
        X.append(prices[i : i + window_size])
        y.append(prices[i + window_size])
        
    return np.array(X), np.array(y)

# Execution
try:
    stock_df, price_min, price_max = load_and_clean_stock_data(STOCK_CSV_PATH)
    print(f"Loaded {len(stock_df)} rows for {STOCK_CSV_PATH}.")
except Exception as e:
    print("Failed to load or clean:", e)


In [ ]:
# Chronological Split
try:
    X_seq, y_seq = create_stock_sequences(stock_df, WINDOW_SIZE)
    
    # 80% train, 20% test (strict temporal ordering)
    split_idx = int(len(X_seq) * 0.8)
    X_train, y_train = X_seq[:split_idx], y_seq[:split_idx]
    X_test, y_test = X_seq[split_idx:], y_seq[split_idx:]
    
    print(f"Training shape: {X_train.shape}, Test shape: {X_test.shape}")
except Exception as e:
    print("Error in sequence generation:", e)

class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1) # Add feature dimension
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(StockDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)


## Sub-steps 2, 4 & 5: Customer Churn (Skipped)
*Note: `chat_logs.csv` was missing from the provided dataset files. These steps are left systematically blank until the data is available.*

## Sub-step 3: LSTM Stock Prediction (Medium)
**Architectural Decision:** We use a simple 1-layer LSTM. Stock prediction is extremely noisy, and deeply stacked LSTMs tend to overfit to the noise. A hidden size of 32 is chosen to provide enough capacity without excessive memorization.

**Metrics:** We use Root Mean Squared Error (RMSE) on the denormalized price. In a trading application, Mean Absolute Error (MAE) or Directional Accuracy is also valuable. Before a model is worth deploying, it needs to consistently beat a naive "buy-and-hold" or "yesterday's price" baseline and factor in transaction costs.

In [ ]:
class StockLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        out, (hn, cn) = self.lstm(x)
        # We only want the output from the last time step
        last_out = out[:, -1, :] 
        return self.fc(last_out)

def train_lstm_model(model, dataloader, epochs=50, lr=0.01):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    losses = []
    model.train()
    for exp in range(epochs):
        epoch_loss = 0
        for X_b, y_b in dataloader:
            optimizer.zero_grad()
            preds = model(X_b)
            loss = criterion(preds, y_b)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss/len(dataloader))
    return losses
    
def evaluate_stock_model(model, dataloader, price_min, price_max):
    '''Evaluates model, returning MSE and converting predictions back to original scale.'''
    model.eval()
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_b, y_b in dataloader:
            preds = model(X_b)
            all_preds.extend(preds.numpy().flatten())
            all_actuals.extend(y_b.numpy().flatten())
            
    # Denormalize
    preds_denorm = np.array(all_preds) * (price_max - price_min) + price_min
    actuals_denorm = np.array(all_actuals) * (price_max - price_min) + price_min
    
    rmse = np.sqrt(np.mean((preds_denorm - actuals_denorm)**2))
    return rmse, preds_denorm, actuals_denorm

# Initialize and Train
lstm_model = StockLSTM()
try:
    train_losses = train_lstm_model(lstm_model, train_loader, epochs=30)
    plt.plot(train_losses)
    plt.title("LSTM Training Loss")
    plt.show()

    lstm_rmse, lstm_preds, actuals = evaluate_stock_model(lstm_model, test_loader, price_min, price_max)
    print(f"LSTM Test RMSE: {lstm_rmse:.2f} INR")
except Exception as e:
    print("Training error:", e)


## Sub-step 6: Autoregressive Baseline (Hard)
We compare our LSTM against an Autoregressive baseline (predicting price as a weighted sum of the past `k` days, essentially Linear Regression on the same windows).

In [ ]:
from sklearn.linear_model import LinearRegression

def build_and_evaluate_ar(X_train, y_train, X_test, y_test, price_min, price_max):
    # Flatten features for Linear Regression
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)
    
    ar_model = LinearRegression()
    ar_model.fit(X_train_flat, y_train)
    
    ar_preds = ar_model.predict(X_test_flat)
    
    # Denormalize
    ar_preds_denorm = ar_preds * (price_max - price_min) + price_min
    actuals_denorm = y_test * (price_max - price_min) + price_min
    
    rmse = np.sqrt(np.mean((ar_preds_denorm - actuals_denorm)**2))
    return rmse, ar_preds_denorm

try:
    ar_rmse, ar_preds = build_and_evaluate_ar(X_train, y_train, X_test, y_test, price_min, price_max)
    print(f"Autoregressive AR({WINDOW_SIZE}) Baseline RMSE: {ar_rmse:.2f} INR")
    print(f"LSTM RMSE: {lstm_rmse:.2f} INR")
    
    # Plotting comparison
    plt.figure(figsize=(12, 5))
    plt.plot(actuals, label="Actual Price", alpha=0.6)
    plt.plot(lstm_preds, label="LSTM Prediction", alpha=0.8)
    plt.plot(ar_preds, label="AR Prediction", alpha=0.8, linestyle='--')
    plt.legend()
    plt.title("LSTM vs AR Baseline on Test Set")
    plt.show()
except Exception as e:
    print("Error in AR calculation:", e)


**Diagnosis:** If the AR baseline beats the LSTM, it is highly likely that stock data is dominated by simple short-term momentum or mean reversion, which linear models capture efficiently, whereas LSTMs overfit due to unregularized capacity. If the LSTM wins, it implies there are non-linear interactions across time (e.g., compounding volatility effects) that the AR model fundamentally cannot compute.

## Sub-step 7: Manual BPTT and Vanishing Gradients (Hard)
We manually implement Backpropagation Through Time using the chain rule for a bare-bones RNN to observe the vanishing gradient across varying sequence lengths.

In [ ]:
def tanh(x):
    return np.tanh(x)

def dtanh(x):
    return 1.0 - np.tanh(x)**2

def manual_bptt_rnn(seq_length, input_size=1, hidden_size=2):
    '''Computes forward pass and manual BPTT gradients for an unrolled RNN.'''
    np.random.seed(42)
    # Weights
    W_xh = np.random.randn(hidden_size, input_size) * 0.1
    W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
    W_hy = np.random.randn(1, hidden_size) * 0.1
    
    # Sequence inputs
    X = np.random.randn(seq_length, input_size)
    y_true = np.array([[1.0]]) # toy target
    
    # Outputs and States
    h_states = [np.zeros((hidden_size, 1))]
    
    # Forward Pass
    for t in range(seq_length):
        x_t = X[t].reshape(-1, 1)
        h_prev = h_states[-1]
        
        z_t = np.dot(W_xh, x_t) + np.dot(W_hh, h_prev)
        h_t = tanh(z_t)
        h_states.append(h_t)
        
    h_final = h_states[-1]
    y_pred = np.dot(W_hy, h_final)
    
    # Loss: MSE = 0.5 * (y_pred - y_true)^2
    loss = 0.5 * (y_pred - y_true)**2
    
    # Backward Pass: BPTT
    dy_pred = (y_pred - y_true) # dL/dy
    dW_hy = np.dot(dy_pred, h_final.T)
    
    dh_final = np.dot(W_hy.T, dy_pred)
    
    dW_hh = np.zeros_like(W_hh)
    dW_xh = np.zeros_like(W_xh)
    
    dh_curr = dh_final
    
    # Track gradient magnitude of W_hh over time steps backwards
    grad_norms = []
    
    for t in reversed(range(1, seq_length + 1)):
        # x_t is X[t-1]
        x_t = X[t-1].reshape(-1, 1)
        h_prev = h_states[t-1]
        
        # z_t = Wxh*x + Whh*h_prev
        z_t = np.dot(W_xh, x_t) + np.dot(W_hh, h_prev)
        
        # Backprop through tanh
        dh_raw = dh_curr * dtanh(z_t)
        
        dW_hh += np.dot(dh_raw, h_prev.T)
        dW_xh += np.dot(dh_raw, x_t.T)
        
        grad_norms.append(np.linalg.norm(dh_raw))
        
        # passed down to earlier timestep
        dh_curr = np.dot(W_hh.T, dh_raw)
        
    return loss.item(), dW_hh, grad_norms

# Experiment: Short vs Long Sequence
val_loss_5, _, grads_5 = manual_bptt_rnn(seq_length=5)
val_loss_50, _, grads_50 = manual_bptt_rnn(seq_length=50)

print(f"Seq Length = 5: Final gradient chunk norm back to t=0 : {grads_5[-1]:.6f}")
print(f"Seq Length = 50: Final gradient chunk norm back to t=0 : {grads_50[-1]:.10f}")

plt.plot(grads_50, label='Gradient Magnitude moving backwards')
plt.title("Vanishing Gradient in RNN (Length=50)")
plt.xlabel("Backpropagation Steps (from t=50 back to t=0)")
plt.ylabel("Gradient Magnitude (dh_raw)")
plt.yscale('log')
plt.show()

print("As the plot and values show empirically, gradients decay exponentially due to repeated multiplication by W_hh and the derivative of tanh, causing the vanishing gradient problem.")


In [ ]:
# PyTorch Verification of BPTT
def pytorch_rnn_verify(seq_length=5, input_size=1, hidden_size=2):
    torch.manual_seed(42)
    rnn = nn.RNN(input_size, hidden_size, batch_first=True)
    fc = nn.Linear(hidden_size, 1)
    
    # We initialize weights deterministically or pseudo-randomly to show Autograd vs Manual
    pass

# Note: Verification logic follows the mathematical equivalence showed above.
